# HITL_ToolNode


In [2]:
from typing import List, Dict, Any, Tuple, Union, Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    ask_human: bool

In [6]:
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """获取某个位置的当前天气。

    Args:
        location (str): 城市名称。

    Returns:
        str: 天气预报。
    """
    return f"{location} 的天气是晴天"
@tool
def request_assistance():
    """将对话升级至专家。如果用户需要的指导超出了助手的能力范围，请使用此功能。"""
    return ""


In [10]:
import dotenv
dotenv.load_dotenv('.env')


True

In [11]:
import os
from pymongo import MongoClient
from langgraph.checkpoint.mongodb import MongoDBSaver

client = MongoClient(host=os.getenv("MONGO_HOST"),
                     port=int(os.getenv("MONGO_PORT")),
                     username=os.getenv("MONGO_USER"),
                     password=os.getenv("MONGO_PASSWORD"))
memory = MongoDBSaver(client)





In [12]:
from langchain_deepseek import ChatDeepSeek

## 构建模型
model = ChatDeepSeek(model="deepseek-chat", 
                     api_key=os.getenv("DEEPSEEK_API_KEY"),
                     base_url=os.getenv("DEEPSEEK_API_BASE"))
llm_with_tools = model.bind_tools([get_weather, request_assistance])

In [ ]:
def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    ask_human = False
    if response.tool_calls and response.tool_calls[0]["name"] == "request_assistance":
        ask_human = True
    return {"messages": [response], "ask_human": ask_human}


In [ ]:
from langgraph.prebuilt import ToolNode

tools_node = ToolNode(tools=[get_weather])


In [ ]:
from langchain_core.messages import ToolMessage, AIMessage

def create_response(response: str, ai_message: AIMessage):
    return ToolMessage(
        content=response,
        tool_call_id=ai_message.tool_calls[0]["id"],
    )
def human_node(state: State):
    new_messages = []
    if not isinstance(state["messages"][-1], ToolMessage):
        new_messages.append(
            create_response(
                "Plan your trip three months in advance and avoid wearing a Real Madrid shirt in Barcelona.",
                state["messages"][-1],
            )
        )
    return {
        "messages": new_messages,
        "ask_human": False,
    }

In [ ]:
from langgraph.prebuilt import tools_condition

def select_next_node(state: State):
    if state["ask_human"]:
        return "human"
    return tools_condition(state)

In [ ]:
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver


graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tools_node)
graph_builder.add_node("human", human_node)
graph_builder.add_conditional_edges(
    "chatbot",
    select_next_node,
    {"human": "human", "tools": "tools", "__end__": "__end__"}
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("human", "chatbot")
graph_builder.set_entry_point("chatbot")
checkpointer = MemorySaver()
graph = graph_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["human"]
)

In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "50"}}
input_message = HumanMessage(
    content="I need some expert advice on how to plan a trip to Barcelona"
)
graph.invoke({"messages": input_message}, config=config)

In [ ]:
graph.invoke({"messages": input_message}, config=config)
